In [ ]:
import os
from transformers import set_seed

os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
set_seed(42, deterministic=True)

import torch

from llama_index.core import PromptTemplate, Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.huggingface import HuggingFaceLLM
from llama_index.core import StorageContext, load_index_from_storage

In [ ]:
query_wrapper_prompt = PromptTemplate(
    "[INST] {query_str} [/INST]"
)  # Taken from Mistral-7B-Instruct-v0.3's chat template
Settings.embed_model = HuggingFaceEmbedding(
    model_name="../../../src/processor/model/weight/bge-base"
)
Settings.llm = HuggingFaceLLM(
    context_window=32768,
    max_new_tokens=1000,
    query_wrapper_prompt=query_wrapper_prompt,
    generate_kwargs={"do_sample": False, "pad_token_id": 2},
    model_kwargs={
        "torch_dtype": torch.bfloat16,
    },
    tokenizer_name="../../../src/processor/model/weight/mistral-7b",
    model_name="../../../src/processor/model/weight/mistral-7b",
    device_map="auto",
    tokenizer_kwargs={"max_length": 32768},
)

In [ ]:
# Path where the index was persisted
persist_dir = f"index/environment"

# Rebuild storage context
storage_context = StorageContext.from_defaults(persist_dir=persist_dir)

In [ ]:
# Load the index from storage
index = load_index_from_storage(storage_context)
query_engine = index.as_query_engine(similarity_top_k=10)

In [ ]:
response = query_engine.query("...")